In [ ]:
import subprocess
import sys
import itertools
import time
from pathlib import Path

# ---------------------------------------------------------
# 1. 探索したいパラメータのグリッドを定義（ここを書き換えてください）
# ---------------------------------------------------------
project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "src").exists():
    project_root = project_root.parent

param_grid = {
    "--model": ["wave1d", "small", "original", "vgg11"],     # 4パターン
    "--batch-size":    [32, 64],                  # 2パターン
    "--epochs":     [100, 200],           # 2パターン
    # 固定したい引数は1つだけのリストにするか、後で固定リストに追加してもOK
}

# 実行するスクリプトのパス
target_script = project_root / "src/metric/LabelClustering.py"
if not target_script.exists():
    raise FileNotFoundError(f"学習スクリプトが見つかりません: {target_script}")

# ---------------------------------------------------------
# 2. 総当たりの組み合わせを作成
# ---------------------------------------------------------
keys = list(param_grid.keys())
values = list(param_grid.values())

# *values でリストを展開して渡すのがコツです
# これで [(0.01, 32, "adam", 0.0), (0.01, 32, "adam", 0.5), ...] のようなリストができます
combinations = list(itertools.product(*values))

total = len(combinations)
print(f"合計 {total} 通りの実験を開始します。\n")

# ---------------------------------------------------------
# 3. ループ実行
# ---------------------------------------------------------
for i, combo in enumerate(combinations, 1):
    
    # 今回のパラメータ引数リストを作成
    # zipを使って、キー("--lr") と 値(0.01) をセットにします
    cmd_args = []
    current_params_str = [] # 表示用
    
    for key, value in zip(keys, combo):
        cmd_args.append(key)
        cmd_args.append(str(value))
        current_params_str.append(f"{key}={value}")

    # 実行コマンドの構築 (sys.executableで現在のconda環境を使用)
    cmd = [sys.executable, str(target_script)] + cmd_args
    
    # 進行状況の表示
    print(f"[{i}/{total}] 実験開始: {', '.join(current_params_str)}")
    
    # --- 実行 ---
    try:
        # ログを見分けやすくしたい場合、実験ごとにログファイルを変える工夫などをここに書くと便利です
        subprocess.run(cmd, check=True, cwd=project_root)
        
    except subprocess.CalledProcessError as e:
        print(f"⚠️ エラー発生 (実験 {i}): {e}")
        # エラーが出ても次の実験に進みたい場合はここで止まらず、
        # continue するか、エラーログを残す処理にします
    
    print("-" * 50)

print("全てのグリッドサーチが完了しました。")

合計 16 通りの実験を開始します。

[1/16] 実験開始: --model=wave1d, --batch-size=32, --epochs=100
2025-12-10 14:45:33,706 - INFO - Hyperparameters:
2025-12-10 14:45:33,707 - INFO - MODEL: wave1d
2025-12-10 14:45:33,707 - INFO - MODEL_DESCRIPTION: 1D Conv encoder (waveform)
2025-12-10 14:45:33,707 - INFO - DATA_SELECTION: loc1-6
2025-12-10 14:45:33,707 - INFO - DATA_CSV_PATH: ./data/processed/datasets/data_1-6.csv
2025-12-10 14:45:33,707 - INFO - MAIN_DATA_DIR: ./data/processed/datasets
2025-12-10 14:45:33,707 - INFO - MEL: OFF
2025-12-10 14:45:33,707 - INFO - SAMPLING_RATE: 16000
2025-12-10 14:45:33,707 - INFO - N_FFT: 1024
2025-12-10 14:45:33,707 - INFO - HOP_LENGTH: 512
2025-12-10 14:45:33,707 - INFO - TEST_DATASET_PERCENTAGE: 0.2
2025-12-10 14:45:33,707 - INFO - BATCH_SIZE: 32
2025-12-10 14:45:33,707 - INFO - EPOCHS: 100
2025-12-10 14:45:33,707 - INFO - LEARNING_RATE: 0.001
2025-12-10 14:45:33,707 - INFO - MARGIN: 0.25
2025-12-10 14:45:33,707 - INFO - GAMMA: 80.0
2025-12-10 14:45:33,707 - INFO - REPRE

Train Epoch 2/100:  23%|██▎       | 7/30 [00:00<00:00, 67.54it/s]

2025-12-10 14:45:47,825 - INFO - Epoch [1/100] train_loss=80.7157 val_loss=80.2696
2025-12-10 14:45:47,827 - INFO - Best model updated (epoch 1, val_loss=80.2696)


Train Epoch 3/100:  27%|██▋       | 8/30 [00:00<00:00, 70.55it/s]

2025-12-10 14:45:48,321 - INFO - Epoch [2/100] train_loss=80.6197 val_loss=80.3472


...

2025-12-10 14:46:31,884 - INFO - Epoch [98/100] train_loss=80.7356 val_loss=80.3602


Train Epoch 100/100:   0%|          | 0/30 [00:00<?, ?it/s]

2025-12-10 14:46:32,361 - INFO - Epoch [99/100] train_loss=80.7351 val_loss=80.3601


Train Epoch 100/100: 100%|██████████| 30/30 [00:00<00:00, 62.38it/s]


2025-12-10 14:46:32,889 - INFO - Epoch [100/100] train_loss=80.7336 val_loss=80.3601
2025-12-10 14:46:32,891 - INFO - Last model checkpoint saved. Best epoch=72
2025-12-10 14:46:33,087 - INFO - 潜在空間の抽出が完了しました。
2025-12-10 14:46:33,087 - INFO - t-SNE による次元削減を開始します。
2025-12-10 14:46:34,237 - INFO - メタデータを ./outputs/2025-12-10/14-45-33/metadata.csv に保存しました。
2025-12-10 14:46:34,238 - INFO - Class 0 平均速度: 51.56
2025-12-10 14:46:34,238 - INFO - Class 1 平均速度: 51.56
2025-12-10 14:46:34,238 - INFO - Class 2 平均速度: 51.56
2025-12-10 14:46:34,238 - INFO - Class 3 平均速度: 51.56
2025-12-10 14:46:34,334 - INFO - 速度による可視化が完了しました。(2D)
2025-12-10 14:46:34,237 - INFO - メタデータを ./outputs/2025-12-10/14-45-33/metadata.csv に保存しました。
2025-12-10 14:46:34,238 - INFO - Class 0 平均速度: 51.56
2025-12-10 14:46:34,238 - INFO - Class 1 平均速度: 51.56
2025-12-10 14:46:34,238 - INFO - Class 2 平均速度: 51.56
2025-12-10 14:46:34,238 - INFO - Class 3 平均速度: 51.56
2025-12-10 14:46:34,334 - INFO - 速度による可視化が完了しました。(2D)
2025-12-10 14:46:34,